# Slug Clustering by Country Coverage

Groups slugs by their country coverage patterns using Jaccard similarity on binary presence matrices.
Missingness is signal — the absence of an indicator for a country is valuable information.

**Phase 1: Model Definition**

In [6]:
using Revise
using InteractiveUtils

includet("phase1/functions/load_phase1.jl")

In [7]:
using CSV, DataFrames

# Load augmented data (uses cached Arrow if available, otherwise runs full pipeline)
df = load_augmented_or_build()
meta_df = CSV.read("data/qog_metadata_plus2.csv", DataFrame)
geo_df = CSV.read("data/ggis_geographic_lookup.csv", DataFrame)
println("Loaded: $(nrow(df)) rows, $(nrow(meta_df)) slugs, $(nrow(geo_df)) countries")

✓ Checksum verified: data/qog_std_ts_jan25_aug.arrow
✓ Loaded: 12391 rows × 2014 cols from data/qog_std_ts_jan25_aug.arrow
  ggis_rowid unique: ✓ | Required columns: ✓ | Missing regions: 0 ✓
Loaded: 12391 rows, 2010 slugs, 207 countries


## Run Clustering Pipeline

Default thresholds — adjust as needed.

In [8]:

# Adjust thresholds:
#     result = run_slug_clustering(df, meta_df, geo_df;
#         presence_min_pct=0.15,    # stricter presence
#         min_sim=0.20,             # tighter clusters
#         k=10                      # fewer edges
#     )

result = run_slug_clustering(df, meta_df, geo_df)

Step 0 — Slug Partitioning
    Total slugs:   2010
    Global (set aside): 616
    Cluster candidates: 1394

Step 1 — Building slug × country presence matrix
    Countries: 200
    Candidate slugs: 1394
    Valid in DataFrame: 1393
    presence_min_pct: 0.1
    presence_max_pct: 0.95
    min_country_coverage: 0.05
    Passing slugs: 1147 (filtered 246 below coverage threshold)
    ⚠️  Near-global slugs flagged: 66
    Matrix: 1147 slugs × 200 countries
    Non-zeros: 95772 (41.75% density)

Step 2 — Jaccard top-k edges
    Slugs: 1147
    Countries: 200
    k: 20, min_sim: 0.1
    Directed edges: 22940
    Similarity range: 0.172 — 1.0
    Median similarity: 0.979

Step 3a — Symmetrize edges
    Directed edges in:   22940
    Undirected edges out: 17552
    Weight range: 0.172 — 1.0
    Bidirectional: 5388 / 17552 (30.7%)

Step 3b — Build weighted graph
    Vertices: 1147
    Edges: 17552
    Connected components: 2
    Component sizes: 1121, 26

Step 3c — Label propagation clustering


(partition = (global_slugs = ["atop_ally", "atop_consult", "atop_defensive", "atop_neutrality", "atop_nonagg", "atop_number", "atop_offensive", "atop_transyr", "bci_bci", "bci_bcistd"  …  "fi_index_pd", "fi_legprop", "fi_legprop_pd", "fi_reg", "fi_reg_pd", "fi_sm", "fi_sm_pd", "fi_sog", "fi_sog_pd", "whr_hap"], cluster_slugs = ["aid_cpnc", "aid_cpsc", "aid_crnc", "aid_crnio", "aid_crsc", "aid_crsio", "aii_acc", "aii_aio", "aii_cilser", "aii_elec"  …  "ident_ccode", "ident_ccode_qog", "ident_ccodealp", "ident_ccodealp_year", "ident_ccodecow", "ident_cname", "ident_cname_qog", "ident_cname_year", "ident_year", "who_roadtrd"]), matrix = (X = sparse([3, 4, 5, 6, 69, 70, 71, 72, 73, 74  …  846, 847, 1139, 1140, 1141, 1142, 1144, 1145, 1146, 1147], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  200, 200, 200, 200, 200, 200, 200, 200, 200, 200], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0  …  1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], 1147, 200), slug_index = ["aid_cpnc", "aid_cpsc", "aid_cr

## Cluster Profiles

In [9]:
result.profiles

Row,cluster_id,n_slugs,n_prefixes,dominant_prefix,dominant_provenance,n_countries,country_coverage_pct,geographic_type,primary_region,primary_region_pct,label
,Int64,Int64,Int64,String7,String,Int64,Float64,String?,String?,Float64?,String
1,1,33,3,sgi,SURVEY,51,25.5,multi-regional,Europe,62.7,Mixed: sgi+ (33 slugs)
2,2,28,4,br,SURVEY,193,96.5,global,Africa,28.0,Global SURVEY (br+)
3,3,58,3,aii,SURVEY,58,29.0,regional,Africa,91.4,Africa-focused (aii+)
4,4,26,1,aii,EXPERT,11,5.5,regional,Africa,100.0,aii
5,5,77,19,iaep,SURVEY,195,97.5,global,Africa,27.7,Global SURVEY (iaep+)
6,6,22,4,bl,SURVEY,156,78.0,sparse,Africa,27.6,Mixed: bl+ (22 slugs)
7,7,65,6,bti,SURVEY,165,82.5,global,Africa,31.5,Global SURVEY (bti+)
8,8,19,4,ident,QoG Standard,200,100.0,global,Africa,27.0,Global QoG Standard (ident+)
9,9,125,11,wdi,SURVEY,194,97.0,global,Africa,27.8,Global SURVEY (wdi+)


## ht_region Alignment

In [10]:
result.ht_validation

Row,cluster_id,best_ht_region,ht_region_jaccard,n_countries
,Int64,Int64,Float64,Int64
1,1,5,0.444,51
2,2,4,0.254,193
3,3,4,0.814,58
4,4,4,0.224,11
5,5,4,0.251,195
6,6,4,0.228,156
7,7,4,0.281,165
8,8,4,0.245,200
9,9,4,0.253,194


## Explore a Specific Cluster

Change `cid` to inspect different clusters.

In [7]:
cid = first(result.profiles.cluster_id)
cluster_slugs = filter(r -> r.cluster_id == cid, result.slug_clusters)
println("Cluster $cid: $(nrow(cluster_slugs)) slugs")
leftjoin(cluster_slugs, meta_df[:, [:slug, :prefix, :label, :provenance]], on=:slug)

LoadError: invalid assignment to constant Main.cluster_slugs. This redefinition may be permitted using the `const` keyword.

## Experiment with Thresholds

In [ ]:
# Tighter clusters: higher min_sim, lower k
# result_tight = run_slug_clustering(df, meta_df, geo_df;
#     presence_min_pct=0.15, min_sim=0.20, k=10)

# Looser clusters: lower min_sim, higher k
# result_loose = run_slug_clustering(df, meta_df, geo_df;
#     presence_min_pct=0.05, min_sim=0.05, k=30)

## Save Results

In [ ]:
# CSV.write("data/slug_clusters.csv", result.slug_clusters)
# println("✅ Saved slug_clusters.csv")